# Part 2: Analysis (B)

All analysis cells are read-only (`q(...)` on `gold.*`). The only exception is the cell that saves a mart, labeled `# SAVE MART`. Each question gets a 2-3 sentence conclusion after the query.

## B1. Do late deliveries lower review scores?
- **Tables:** `fact_orders` (A), `fact_reviews`
- **Method:** average `review_score_avg` for late vs on-time orders; share of 1-2 star reviews by delay bucket (on time, 1-3 days late, 4-7 days, 8+ days). Optional: Mann-Whitney test in pandas/scipy.
- **Output:** DAX measures `Avg Review Score` by `is_late`; delay buckets as a calculated column.
- **Conclusion:** _write after running_

## B2. Which sellers perform best and worst?
- **Tables:** `fact_order_items`, `fact_orders`, `fact_reviews`, `dim_sellers`
- **Method:** per seller: revenue, orders, average review score, late-order share, average delivery days. Ignore sellers with very few orders (for example under 10) when ranking.
- **Output:** saved mart `gold.mart_seller_performance`.
- **Conclusion:** _write after running_

## B3. Which product categories drive revenue and satisfaction?
- **Tables:** `fact_order_items`, `dim_products`, `fact_reviews`
- **Method:** revenue, items sold, average price and average review score per `category_name_english`; top 10 by revenue; categories with the lowest scores.
- **Output:** DAX measures by `dim_products[category_name_english]`.
- **Conclusion:** _write after running_

## B4. Does rain affect orders? (São Paulo only)
- **Tables:** `fact_orders` (A), `dim_customers` (A), `dim_date`, `weather`
- **Method:** daily orders and revenue on rainy vs dry days (`precipitation > 0`, or above a chosen threshold); also late-delivery share on rainy vs dry days. Only `state = 'SP'`.
- **Output:** saved mart `gold.mart_daily_sales_weather`.
- **Limitation:** weather covers São Paulo only; mark every visual "SP only".
- **Conclusion:** _write after running_

## B5. Do holidays change sales?
- **Tables:** `fact_orders`, `dim_date`
- **Method:** average daily orders and revenue on holidays vs normal days; also the days just before and after major holidays. Compare weekdays with weekdays to avoid weekend bias.
- **Output:** DAX measures by `dim_date[is_holiday]`.
- **Conclusion:** _write after running_

## B6. (Optional) Does the exchange rate matter?
- **Tables:** `fact_order_items`, `dim_date`
- **Method:** monthly revenue in BRL and in USD (`price * usd_per_brl`); does USD revenue move differently from BRL revenue?
- **Output:** DAX measure `Revenue USD`.
- **Conclusion:** _write after running_

In [7]:
import pandas as pd
from sqlalchemy import text
import sys

sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")
from db import get_engine

engine = get_engine()

## B1

Biz burada sifarişlərin gecikmə vaxtında verilən qiymətləndirməyə təsir edir ya yox ona baxırıq. Nəticə olaraq vaxtında çatdırılmış sifarişlərə verilən qiymətləndirmə ilə gecikmiş sifarişlərin qiymətləndirilməsinin ortalamasına baxırıq ..Qiymətləndirmənin səbəbi olmasada təsir etdiyi görülür 

In [9]:

import pandas as pd

df = pd.read_sql("""
    SELECT CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 'Gecikib'
        ELSE 'Vaxtında'
        END AS delivery_status,
        AVG(CAST(r.review_score AS FLOAT)) AS avg_review_score
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r
    ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 'Gecikib'
        ELSE 'Vaxtında'
        END """, engine)

df

,delivery_status,avg_review_score
0,Vaxtinda,4.293718
1,Gecikib,2.566494


Verilən 1 və 2 qiymətləndirmələrin neçə faizinin gecikməyə görə olduğuna baxırıq və görürükki 
vaxtında çatdırılma təxminən hər 100 rəydən 9-u narazıdır.
Gecikmiş çatdırılma təxminən hər 100 rəydən 54-ü narazıdır və buda az verilən qiymətləndirmənin böyük bir hissəsinin hardasa yarısı qədər gecikmənin payında düşür   

In [13]:
df = pd.read_sql("""
    SELECT CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) <= 0
        THEN 'Vaxtinda'
        ELSE 'Gecikib'
        END AS delivery_status,
        COUNT(*) AS total_reviews,
        AVG(CASE WHEN r.review_score <= 2 THEN 1.0 ELSE 0.0 END) * 100
            AS unhappy_review_pct
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY
        CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) <= 0
        THEN 'Vaxtinda'
        ELSE 'Gecikib'
        END 
""", engine)

df

,delivery_status,total_reviews,unhappy_review_pct
0,Vaxtinda,89944,9.2791
1,Gecikib,6409,62.4122


Təxmini və faktiki çatdırılma tarixlərini müqayisə etdik. Təqvim günləri üzrə gecikmə qaydasını tətbiq edərək vaxtında çatdırılan və gecikən sifarişləri ayırdıq. Daha sonra bu qruplar üzrə müştərilərin orta rəy balını və narazı rəylərin faizini hesabladıq. Məqsədimiz çatdırılma gecikmələrinin müştəri məmnuniyyəti ilə əlaqəsini araşdırmaqdır.Bu nəticədən görə bilərik ki gecikmə vaxtı artdıqca aşağı rəylərdə artaraq davam edir.  

In [14]:
df = pd.read_sql("""
    SELECT CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) <= 0
        THEN 'Vaxtında'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 1 AND 3
        THEN '1-3 gun gecikib'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 4 AND 7
        THEN '4-7 gun gecikib'
        ELSE '8+ gun gecikib'
        END AS delay_bucket,
        COUNT(*) AS total_reviews,
        AVG(CASE WHEN r.review_score <= 2 THEN 1.0 ELSE 0.0 END) * 100
        AS unhappy_review_pct
    FROM silver.orders o
    INNER JOIN gold.fact_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
    GROUP BY CASE WHEN DATEDIFF(DAY, o.order_estimated_delivery_date, o.order_delivered_customer_date) <= 0
        THEN 'Vaxtında'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 1 AND 3
        THEN '1-3 gun gecikib'
        WHEN DATEDIFF(DAY, o.order_estimated_delivery_date,o.order_delivered_customer_date) BETWEEN 4 AND 7
        THEN '4-7 gun gecikib'
        ELSE '8+ gun gecikib'
        END
""", engine)

df

,delay_bucket,total_reviews,unhappy_review_pct
0,4-7 gun gecikib,1756,67.6537
1,Vaxtinda,89944,9.2791
2,8+ gun gecikib,2797,79.1919
3,1-3 gun gecikib,1856,32.1659
